Prediction of Clinical Trial Attrition — Dataset Construction
HIDS-6001, Massive Health Data Fundamentals, Fall 2024
Team 3a: Stephanie Araki, Pari Shah, Samantha Zocher

Pulls trial-level data from the ClinicalTrials.gov API for 1,325 trials with known
attrition rates, merges in RUCA (Rural-Urban Commuting Area) geographic categories,
and builds the feature matrix used for attrition prediction.

In [ ]:
import time
import requests
import pandas as pd

In [ ]:
CTGOV_API_BASE = "https://clinicaltrials.gov/api/v2/studies"

### `fetch_trial_record`

In [ ]:
def fetch_trial_record(nct_id: str) -> dict:
    """Pull one trial's structured fields from the ClinicalTrials.gov v2 API."""
    resp = requests.get(f"{CTGOV_API_BASE}/{nct_id}", timeout=15)
    resp.raise_for_status()
    data = resp.json()
    protocol = data.get("protocolSection", {})

    identification = protocol.get("identificationModule", {})
    status = protocol.get("statusModule", {})
    design = protocol.get("designModule", {})
    eligibility = protocol.get("eligibilityModule", {})
    conditions = protocol.get("conditionsModule", {})
    sponsor = protocol.get("sponsorCollaboratorsModule", {})
    locations = protocol.get("contactsLocationsModule", {}).get("locations", [])

    return {
        "nct_id": nct_id,
        "brief_title": identification.get("briefTitle"),
        "official_title": identification.get("officialTitle"),
        "organization_name": sponsor.get("leadSponsor", {}).get("name"),
        "organization_class": sponsor.get("leadSponsor", {}).get("class"),
        "overall_status": status.get("overallStatus"),
        "start_date": status.get("startDateStruct", {}).get("date"),
        "end_date": status.get("completionDateStruct", {}).get("date"),
        "study_type": design.get("studyType"),
        "phase": (design.get("phases") or [None])[0],
        "primary_purpose": design.get("designInfo", {}).get("primaryPurpose"),
        "actual_enrollment": design.get("enrollmentInfo", {}).get("count"),
        "condition": (conditions.get("conditions") or [None])[0],
        "min_age_eligible": eligibility.get("minimumAge"),
        "max_age_eligible": eligibility.get("maximumAge"),
        "gender_eligible": eligibility.get("sex"),
        "locations_count": len(locations),
        "locations": [loc.get("city") for loc in locations],
    }

### `build_trial_dataset`

In [ ]:
def build_trial_dataset(attrition_csv: str, ruca_csv: str, out_csv: str) -> pd.DataFrame:
    """Fetch API data for every NCT ID in the attrition dataset, merge with RUCA
    geographic categories, and write the combined feature matrix to disk."""
    attrition_df = pd.read_csv(attrition_csv)  # columns: nct_id, dropout_percentage_all
    ruca_df = pd.read_csv(ruca_csv)  # zip-code-level rural/urban classification

    records = []
    for nct_id in attrition_df["nct_id"]:
        try:
            records.append(fetch_trial_record(nct_id))
        except requests.RequestException as e:
            print(f"Skipping {nct_id}: {e}")
        time.sleep(0.1)  # basic rate limiting

    trials_df = pd.DataFrame(records)
    merged = trials_df.merge(attrition_df, on="nct_id", how="inner")

    # Most-common RUCA classification (urban vs. non-urban) across a trial's locations
    merged["ruca_binary_most_common"] = merged["locations"].apply(
        lambda cities: _majority_ruca_class(cities, ruca_df)
    )

    merged.to_csv(out_csv, index=False)
    return merged

### `_majority_ruca_class`

In [ ]:
def _majority_ruca_class(cities: list, ruca_df: pd.DataFrame) -> str:
    """Look up each trial location in the RUCA table and return the majority
    urban/non-urban classification across all recruitment sites."""
    if not cities:
        return "Undefined"
    matches = ruca_df[ruca_df["city"].isin(cities)]
    if matches.empty:
        return "Undefined"
    return matches["ruca_class"].mode().iloc[0]

### Run

In [ ]:
if __name__ == "__main__":
    build_trial_dataset(
        attrition_csv="data/ct_attrition_dataset.csv",
        ruca_csv="data/ruca_zip_codes.csv",
        out_csv="data/nct_features_all.csv",
    )